# Phase 4: QLoRA + DPO Fine-Tuning (Kaggle T4, free)

Runs `src/finetune/train_sft.py` -> `src/finetune/train_dpo.py` -> `src/finetune/merge_adapter.py` against the `logcat-intelligence-engine` repo, on Kaggle's free T4 GPU (30 hr/week).

**Before running:** Settings (right panel) -> Accelerator -> GPU T4 x2 (or x1). Also add Secrets: `HF_TOKEN`, `WANDB_API_KEY` (both free, see the repo's `capstone-build-requirements.md` account checklist).

**Known caveats — read before interpreting results:**
- The current `data/sft/train.jsonl` has only **7 records** (2 val) and `data/dpo/train.jsonl` has **1 pair** — nowhere near enough to meaningfully shift model behavior. This run mainly proves the pipeline works end to end, not that fine-tuning improved accuracy. See `docs/components/04-training-data-pipeline.md`.
- The Phase 3 eval baseline (90% category accuracy) is contaminated — the RAG seed corpus and eval tasks were written by the same author with overlapping phrasing, so it will NOT show a fair before/after fine-tuning delta. See `docs/EVAL_SET_IMPROVEMENT_PLAN.md` for the fix, ideally done before drawing conclusions from any eval run against a fine-tuned checkpoint.
- `src/finetune/train_sft.py` and `train_dpo.py` were written and version-checked (imports, `SFTConfig`/`DPOConfig` field names) against `trl==1.9.0` locally, but the actual GPU training has **never been executed** — this notebook is the first real run. If Kaggle's preinstalled `trl`/`transformers` versions differ, you may need to adjust kwargs; run `pip show trl transformers` first to compare.

In [ ]:
!nvidia-smi

## 1. Clone the repo and install cloud-only dependencies

Kaggle's base image already has a CUDA-enabled `torch` installed — avoid reinstalling it via `requirements.txt` (which could pull in a CPU/mismatched build). Install everything else plus `bitsandbytes` explicitly.

In [ ]:
!git clone https://github.com/SatyamKumar02/logcat-intelligence-engine.git
%cd logcat-intelligence-engine

In [ ]:
!pip install -q transformers peft trl accelerate datasets sentence-transformers faiss-cpu wandb jsonlines python-dotenv bitsandbytes

## 2. Set credentials from Kaggle Secrets

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["PYTHONPATH"] = "."

## 3. Validate the data pipeline (no GPU needed, ~1 min)

Already run and verified locally on the Mac (see `CONTEXT.md`) — re-running here just confirms nothing broke in transit (git clone, dependency versions).

In [ ]:
!python scripts/validate_finetune_data.py

## 4. QLoRA SFT training

With only 7 training records this will finish in well under a minute of actual GPU time — most of the wall-clock here is model download (~15GB, one-time) and quantized model load.

In [ ]:
!python src/finetune/train_sft.py

## 5. DPO training (optional — only 1 preference pair right now)

Safe to run (proves the pipeline works), but do not expect a meaningful behavior change from a single pair. Skip this cell if you'd rather wait until more human-reviewed corrections accumulate (see `docs/components/04-training-data-pipeline.md`).

In [ ]:
!python src/finetune/train_dpo.py

## 6. Merge the LoRA adapter for deployment

Runs on CPU. Point `adapter_path` at `outputs/dpo-diagnostic` instead of `outputs/qlora-diagnostic` if you ran the DPO step and want the DPO-refined adapter merged instead of the plain SFT one.

In [ ]:
!python src/finetune/merge_adapter.py

## 7. Persist the merged model as Kaggle notebook output

Anything under `/kaggle/working/` is saved when the notebook session ends and can be downloaded or turned into a Kaggle Dataset for the Phase 5 deployment step.

In [ ]:
!mkdir -p /kaggle/working/merged-diagnostic-v1
!cp -r outputs/merged-diagnostic-v1/* /kaggle/working/merged-diagnostic-v1/
!ls -lh /kaggle/working/merged-diagnostic-v1/

## Next steps

1. Fix the eval set (`docs/EVAL_SET_IMPROVEMENT_PLAN.md`) before trusting any accuracy comparison.
2. Run `scripts/run_eval.py` against both the base model and this merged checkpoint (once vLLM/an inference endpoint is available — Phase 5) for a real before/after number.
3. Once more traces + human corrections accumulate (Phase 2/6 flywheel), re-run this notebook — the current 7-record SFT set is a pipeline proof, not a real training run.